<a href="https://colab.research.google.com/github/NicoMasss/MediAgent-RL/blob/main/MediAgent-RL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install gymnasium stable-baselines3 torch
!pip install tqdm

import gymnasium as gym
from gymnasium import spaces
import numpy as np
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
import os

# Importações de DRL da stable-baselines3
from stable_baselines3 import DQN, A2C, PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 16.7 MB/s eta 0:00:00


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
# Definição do Ambiente Gymnasium
class PacienteVirtualEnvFA(gym.Env):
    """
    Ambiente Gymnasium customizado para simular o tratamento de um paciente,
    ajustado para Deep Reinforcement Learning (DRL) com espaços de observação contínuos.

    Estado: [Infecção, Febre, Dor]
    - Infecção, Febre, Dor: Valores de 0.0 (melhor) a 2.0 (pior)

    Ações (Discretas):
    - 0: Administrar Antibiótico
    - 1: Administrar Antitérmico
    - 2: Administrar Analgésico
    - 3: Aguardar
    """
    metadata = {'render_modes': ['human']}

    def __init__(self):
        super().__init__()

        # Definição do espaço de ações (4 ações discretas)
        self.action_space = spaces.Discrete(4)

        self.observation_space = spaces.Box(
            low=0.0,
            high=2.0,
            shape=(3,),
            dtype=np.float32
        )

        self._build_transition_probabilities()

        self.CURADO_STATE = np.array([0.0, 0.0, 0.0], dtype=np.float32)
        self.OBITO_STATE = np.array([3.0, 3.0, 3.0], dtype=np.float32)
        self.state = self.CURADO_STATE.copy()

    def _to_idx(self, state_tuple):
        infection, fever, pain = state_tuple
        return int(infection) * 9 + int(fever) * 3 + int(pain)

    def _to_tuple(self, state_idx):
        if state_idx >= 27: return None
        infection = state_idx // 9
        fever = (state_idx % 9) // 3
        pain = state_idx % 3
        return (infection, fever, pain)

    def _build_transition_probabilities(self):
        self.P_idx = {s: {a: [] for a in range(self.action_space.n)} for s in range(29)}

        for state_idx in range(27):
            infection, fever, pain = self._to_tuple(state_idx)

            if infection == 2 and fever == 2:
                 self.P_idx[state_idx][0] = [(0.1, 28, -100, True)]

            if infection == 0 and fever == 0 and pain == 0:
                for a in range(4):
                    self.P_idx[state_idx][a] = [(1.0, 27, 100, True)]
                continue

            symptom_penalty = -5 if (fever == 2 or pain == 2) else 0

            # Ação 0: Antibiótico
            cost = -1
            next_i_good = max(0, infection - 1)
            next_f_bad = min(2, fever + 1)
            self.P_idx[state_idx][0] = [
                (0.7, self._to_idx((next_i_good, fever, pain)), cost + symptom_penalty, False),
                (0.2, state_idx, cost + symptom_penalty, False),
                (0.1, self._to_idx((infection, next_f_bad, pain)), cost + symptom_penalty, False)
            ]

            # Ação 1: Antitérmico
            cost = -1
            next_f_good = max(0, fever - 1)
            self.P_idx[state_idx][1] = [
                (0.8, self._to_idx((infection, next_f_good, pain)), cost + symptom_penalty, False),
                (0.2, state_idx, cost + symptom_penalty, False)
            ]

            # Ação 2: Analgésico
            cost = -1
            next_p_good = max(0, pain - 1)
            self.P_idx[state_idx][2] = [
                (0.8, self._to_idx((infection, fever, next_p_good)), cost + symptom_penalty, False),
                (0.2, state_idx, cost + symptom_penalty, False)
            ]

            # Ação 3: Aguardar
            cost = -2
            next_i_bad = min(2, infection + 1)
            self.P_idx[state_idx][3] = [
                (0.6, self._to_idx((next_i_bad, fever, pain)), cost + symptom_penalty, False),
                (0.4, state_idx, cost + symptom_penalty, False)
            ]

        for idx in [27, 28]:
             for a in range(4):
                self.P_idx[idx][a] = [(1.0, idx, 0, True)]

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        initial_state_tuple = (1.0, 1.0, 1.0)
        self.state = np.array(initial_state_tuple, dtype=np.float32)
        return self.state, {}

    def step(self, action):

        if np.array_equal(self.state, self.CURADO_STATE):
            return self.state, 0.0, True, False, {}

        current_state_idx = self._to_idx(tuple(self.state.astype(int)))

        transitions = self.P_idx[current_state_idx][action]

        if current_state_idx >= 27:
             return self.state, 0.0, True, False, {}

        probs = [t[0] for t in transitions]

        chosen_idx = np.random.choice(len(transitions), p=probs)
        _, next_state_idx, reward, terminated = transitions[chosen_idx]

        if next_state_idx == 27:
            next_state = self.CURADO_STATE
            terminated = True
            reward = 100
        elif next_state_idx == 28:
            next_state = self.OBITO_STATE
            terminated = True
            reward = -100
        else:
            next_state = np.array(self._to_tuple(next_state_idx), dtype=np.float32)

        self.state = next_state
        truncated = False

        return self.state, float(reward), terminated, truncated, {}

    def render(self):
        status_map = {0: "Controlada/Normal/Sem Dor", 1: "Ativa/Média/Moderada", 2: "Grave/Alta/Intensa"}

        if np.array_equal(self.state, self.CURADO_STATE):
            print("Estado do Paciente: CURADO")
        elif np.array_equal(self.state, self.OBITO_STATE):
            print("Estado do Paciente: ÓBITO")
        else:
            i, f, d = self.state.astype(int)
            print(f"Estado do Paciente: (Infecção: {status_map[i]}, Febre: {status_map[f]}, Dor: {status_map[d]})")


# Registrar o ambiente de FA
gym.register(
    id='PacienteVirtualFA-v0',
    entry_point=PacienteVirtualEnvFA,
    max_episode_steps=50,
)

In [4]:
TOTAL_TIMESTEPS = 100000
EVAL_FREQ = 5000
EVAL_EPISODES = 100
N_ENVS = 1
LOG_DIR = "./temp_logs/"

POLICY_KWARGS = dict(net_arch=[dict(pi=[32, 32], vf=[32, 32])])

GAMMA_OPTIMIZED = 0.90
LEARNING_RATE_DQN = 5e-4

env_train = make_vec_env('PacienteVirtualFA-v0', n_envs=N_ENVS, seed=0)
env_eval = DummyVecEnv([lambda: gym.make('PacienteVirtualFA-v0')])


def train_and_evaluate_model(model_class, policy, algorithm_name, timesteps=TOTAL_TIMESTEPS):
    """Função para treinar e avaliar um algoritmo da SB3."""
    print(f"\n--- Treinando {algorithm_name} (Function Approximation) ---")

    eval_callback = EvalCallback(
        env_eval,
        best_model_save_path=f'./best_model_{algorithm_name}/',
        log_path=LOG_DIR,
        eval_freq=EVAL_FREQ // N_ENVS,
        n_eval_episodes=EVAL_EPISODES,
        deterministic=True,
        render=False
    )

    if model_class == DQN:
        model = model_class(
            policy, env_train, verbose=0, tensorboard_log=LOG_DIR,
            learning_rate=LEARNING_RATE_DQN,
            gamma=GAMMA_OPTIMIZED,
            policy_kwargs=dict(net_arch=[32, 32])
        )
    else:
        # A2C e PPO
        model = model_class(
            policy, env_train, verbose=0, tensorboard_log=LOG_DIR,
            gamma=GAMMA_OPTIMIZED,
            policy_kwargs=POLICY_KWARGS
        )

    # Treinamento
    model.learn(total_timesteps=timesteps, callback=eval_callback)

    # Carrega o melhor modelo encontrado
    try:
        model = model_class.load(f'./best_model_{algorithm_name}/best_model', env=env_eval)
    except:
        print(f"Aviso: Não foi possível carregar o modelo ótimo para {algorithm_name}. Usando o modelo final.")

    # Avaliação final do melhor modelo (1000 episódios)
    mean_reward, cure_rate = evaluate_policy_sb3(model, env_eval, n_eval_episodes=1000)

    return mean_reward, cure_rate

def evaluate_policy_sb3(model, eval_env, n_eval_episodes=10):
    """Função de avaliação customizada para extrair taxa de cura e recompensa média."""

    episode_rewards = []
    cure_count = 0

    for _ in range(n_eval_episodes):
        obs = eval_env.reset()
        done = False
        episode_reward = 0

        while not done:
            action, _states = model.predict(obs, deterministic=True)
            obs, reward, done_tuple, info = eval_env.step(action)
            done = done_tuple[0]
            episode_reward += reward[0]

        episode_rewards.append(episode_reward)

        CURADO_STATE_BASE = eval_env.unwrapped.envs[0].unwrapped.CURADO_STATE

        if np.array_equal(obs[0], CURADO_STATE_BASE):
            cure_count += 1

    mean_reward = np.mean(episode_rewards)
    cure_rate = (cure_count / n_eval_episodes) * 100

    return mean_reward, cure_rate


# --- Execução dos Experimentos ---
results = {}

# 1. DQN (Deep Q-Network)
mean_reward_dqn, cure_rate_dqn = train_and_evaluate_model(DQN, "MlpPolicy", "DQN")
results["DQN"] = {"Recompensa Média": mean_reward_dqn, "Taxa de Cura (%)": cure_rate_dqn}

# 2. A2C (Advantage Actor-Critic)
mean_reward_a2c, cure_rate_a2c = train_and_evaluate_model(A2C, "MlpPolicy", "A2C")
results["A2C"] = {"Recompensa Média": mean_reward_a2c, "Taxa de Cura (%)": cure_rate_a2c}

# 3. PPO (Proximal Policy Optimization)
mean_reward_ppo, cure_rate_ppo = train_and_evaluate_model(PPO, "MlpPolicy", "PPO")
results["PPO"] = {"Recompensa Média": mean_reward_ppo, "Taxa de Cura (%)": cure_rate_ppo}


print("\n" + "="*70)
print("Resultados Finais - Aproximação de Função (Grau B)")
print("="*70)

# Estrutura a tabela de resultados
print(f"{'Algoritmo':<15} | {'Recompensa Média':<20} | {'Taxa de Cura (%)':<20}")
print("-" * 70)

for algo, metrics in results.items():
    reward_str = f"{metrics['Recompensa Média']:.2f}"
    cure_str = f"{metrics['Taxa de Cura (%)']:.2f}%"

    print(f"{algo:<15} | {reward_str:<20} | {cure_str:<20}")

print("-" * 70)

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:729: UserWarning: WARN: The environment is being initialised with render_mode='rgb_array' that is not in the possible render_modes (['human']).
  logger.warn(



--- Treinando DQN (Function Approximation) ---


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Eval num_timesteps=5000, episode_reward=-50.00 +/- 0.00
Episode length: 50.00 +/- 0.00
New best mean reward!
Eval num_timesteps=10000, episode_reward=-50.00 +/- 0.00
Episode length: 50.00 +/- 0.00
Eval num_timesteps=15000, episode_reward=-4.12 +/- 1.33
Episode length: 5.12 +/- 1.33
New best mean reward!
Eval num_timesteps=20000, episode_reward=-4.10 +/- 1.40
Episode length: 5.10 +/- 1.40
New best mean reward!
Eval num_timesteps=25000, episode_reward=-3.96 +/- 1.29
Episode length: 4.96 +/- 1.29
New best mean reward!
Eval num_timesteps=30000, episode_reward=-4.07 +/- 1.37
Episode length: 5.07 +/- 1.37
Eval num_timesteps=35000, episode_reward=-4.07 +/- 1.36
Episode length: 5.07 +/- 1.36
Eval num_timesteps=40000, episode_reward=-4.19 +/- 1.71
Episode length: 5.19 +/- 1.71
Eval num_timesteps=45000, episode_reward=-3.92 +/- 1.25
Episode length: 4.92 +/- 1.25
New best mean reward!
Eval num_timesteps=50000, episode_reward=-5.17 +/- 3.82
Episode length: 5.27 +/- 1.38
Eval num_timesteps=55000, e

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/policies.py:486: UserWarning: As shared layers in the mlp_extractor are removed since SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])
  warnings.warn(


Eval num_timesteps=5000, episode_reward=-4.20 +/- 1.52
Episode length: 5.20 +/- 1.52
New best mean reward!
Eval num_timesteps=10000, episode_reward=-4.20 +/- 1.52
Episode length: 5.20 +/- 1.52
Eval num_timesteps=15000, episode_reward=-4.06 +/- 1.62
Episode length: 5.06 +/- 1.62
New best mean reward!
Eval num_timesteps=20000, episode_reward=-4.22 +/- 2.01
Episode length: 5.22 +/- 2.01
Eval num_timesteps=25000, episode_reward=-4.09 +/- 1.37
Episode length: 5.09 +/- 1.37
Eval num_timesteps=30000, episode_reward=-4.29 +/- 2.22
Episode length: 4.89 +/- 1.18
Eval num_timesteps=35000, episode_reward=-4.46 +/- 2.55
Episode length: 4.96 +/- 1.18
Eval num_timesteps=40000, episode_reward=-5.25 +/- 4.21
Episode length: 5.25 +/- 1.53
Eval num_timesteps=45000, episode_reward=-5.49 +/- 5.77
Episode length: 5.04 +/- 1.31
Eval num_timesteps=50000, episode_reward=-5.43 +/- 5.01
Episode length: 5.08 +/- 1.18
Eval num_timesteps=55000, episode_reward=-5.25 +/- 4.35
Episode length: 5.15 +/- 1.42
Eval num_ti